# VibroFlow AI - Deep Learning sur les Défauts de Roulements (CWRU)

Ce notebook détaille l'entraînement du modèle CNN pour la classification des signatures vibratoires provenant du dataset CWRU.

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Robust module path setup
def setup_sys_path():
    curr = os.getcwd()
    # Climb up until we find 'src'
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'src')):
            if curr not in sys.path: sys.path.insert(0, curr)
            src = os.path.join(curr, 'src')
            if src not in sys.path: sys.path.insert(0, src)
            return curr
        curr = os.path.dirname(curr)
    return None

project_root = setup_sys_path()
print(f"Project root: {project_root}")

from data.loader import CWRUBearingDataLoader
from models.deep_learning import CNN1D
# Use project_root for data path
data_path = os.path.join(project_root, 'dataset1')
loader = CWRUBearingDataLoader(data_path)
loader = CWRUBearingDataLoader('../dataset1')
X, y = loader.load_preprocessed_npz()

print(f"Shape des données : {X.shape}")
print(f"Classes uniques : {np.unique(y)}")

## 1. Visualisation des Signaux Vibratoires

Signaux bruts pour différentes classes de défauts.

In [ ]:
classes = np.unique(y)
plt.figure(figsize=(15, 10))

for i, cls in enumerate(classes[:4]):
    idx = np.where(y == cls)[0][0]
    plt.subplot(4, 1, i+1)
    plt.plot(X[idx].flatten())
    plt.title(f'Classe : {cls}')
    plt.grid(True)

plt.tight_layout()
plt.show()

## 2. Analyse Fréquentielle (FFT)

Le domaine fréquentiel est crucial pour identifier les fréquences de défaut caractéristiques.

In [ ]:
def plot_fft(signal, fs=48000):
    n = len(signal)
    freq = np.fft.fftfreq(n, 1/fs)[:n//2]
    magnitude = np.abs(np.fft.fft(signal))[:n//2]
    plt.plot(freq, magnitude)

plt.figure(figsize=(12, 6))
plot_fft(X[0].flatten())
plt.title('Spectre de Fréquence - Exemple de Signal')
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude')
plt.show()

## 3. Validation du Modèle CNN

Visualisons les performances du modèle entraîné à l'aide d'une matrice de confusion.

In [ ]:
# Chargement du modèle (si entraîné)
try:
    # Note: On utilise ici un exemple simulé si le fichier pth n'est pas chargé directement dans le notebook
    from models.deep_learning import DeepLearningTrainer
    
    input_dim = 32*32 # X est (4600, 32, 32) dans cet exemple
    model = CNN1D(input_size=input_dim, num_classes=10)
    trainer = DeepLearningTrainer(model)
    
    # trainer.load('../models/cwru_cnn_model.pth')
    
    print("Modèle prêt pour l'inférence.")
except Exception as e:
    print(f"Erreur ou modèle non trouvé : {e}")